# ResNet-18 이미지 분류 예제

Kaggle Notebook 환경에서 Hugging Face `transformers`와 `datasets`를 사용해 `microsoft/resnet-18` 모델로 샘플 이미지를 분류합니다.

이 노트북은 다음 흐름으로 구성되어 있습니다.

1. 필요한 라이브러리 설치 및 임포트
2. `datasets` 라이브러리에서 샘플 이미지 로드
3. `AutoImageProcessor`로 이미지 전처리
4. PyTorch 기반 ResNet-18 추론
5. 가장 높은 확률의 클래스와 top-10 결과 출력

In [ ]:
# Kaggle 환경에 필요한 패키지가 없을 수 있으므로 먼저 설치합니다.
# 이미 설치되어 있다면 빠르게 넘어갑니다.
%pip install -q transformers datasets pillow matplotlib

In [ ]:
# 기본 라이브러리 임포트
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification

# 실행 장치를 선택합니다. Kaggle에서 GPU를 켜면 cuda가 사용됩니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

## 1. datasets 라이브러리에서 샘플 이미지 불러오기

`huggingface/cats-image` 데이터셋은 작은 예제 이미지 데이터셋이라 이미지 분류 추론 테스트에 사용하기 좋습니다.

In [ ]:
# Hugging Face datasets에서 샘플 이미지 데이터셋을 불러옵니다.
dataset = load_dataset("huggingface/cats-image")

# test split의 첫 번째 이미지를 사용합니다.
# PIL 이미지를 RGB 형식으로 변환해 모델 입력에 맞춥니다.
image = dataset["test"][0]["image"].convert("RGB")

# 추론에 사용할 이미지를 확인합니다.
plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title("Sample Image")
plt.show()

## 2. microsoft/resnet-18 모델과 AutoImageProcessor 로드

`AutoImageProcessor`는 모델이 학습될 때 사용한 이미지 크기, 정규화 평균과 표준편차 등을 자동으로 불러와 동일한 방식으로 전처리합니다.

In [ ]:
# 사용할 사전학습 모델 이름입니다.
model_name = "microsoft/resnet-18"

# 이미지 전처리기와 이미지 분류 모델을 불러옵니다.
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

# 모델을 선택한 장치로 이동하고 평가 모드로 전환합니다.
model.to(device)
model.eval()

print("모델과 이미지 프로세서 로드 완료")

## 3. AutoImageProcessor로 이미지 전처리

전처리 결과는 PyTorch 텐서 형태로 반환되며, 모델 입력에 바로 사용할 수 있습니다.

In [ ]:
# AutoImageProcessor가 resize, center crop, normalize 등을 자동으로 수행합니다.
inputs = processor(images=image, return_tensors="pt")

# 입력 텐서를 모델이 올라간 장치와 동일한 장치로 이동합니다.
inputs = {key: value.to(device) for key, value in inputs.items()}

# 전처리된 텐서 크기를 확인합니다.
for key, value in inputs.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}")

## 4. PyTorch 기반 추론 수행

`torch.no_grad()`를 사용하면 추론 과정에서 그래디언트를 저장하지 않아 메모리를 아낄 수 있습니다.

In [ ]:
# PyTorch 기반으로 이미지 분류 추론을 수행합니다.
with torch.no_grad():
    outputs = model(**inputs)

# logits shape은 [batch_size, num_classes]입니다.
# 샘플 이미지 1장만 사용하므로 [0]으로 첫 번째 배치의 확률만 가져옵니다.
logits = outputs.logits
probabilities = torch.softmax(logits, dim=-1)[0]

print(f"확률 텐서 크기: {tuple(probabilities.shape)}")
print(f"클래스 개수: {probabilities.numel()}")

## 5. 가장 높은 확률의 클래스 출력

모델 설정에 포함된 `id2label` 매핑을 사용해 클래스 번호를 사람이 읽을 수 있는 라벨로 변환합니다.

In [ ]:
# 가장 높은 확률을 가진 클래스 인덱스를 구합니다.
predicted_class_id = int(torch.argmax(probabilities).item())
predicted_label = model.config.id2label[predicted_class_id]
predicted_probability = float(probabilities[predicted_class_id].item())

print("가장 높은 확률의 예측 결과")
print(f"class id   : {predicted_class_id}")
print(f"label      : {predicted_label}")
print(f"probability: {predicted_probability:.4f} ({predicted_probability * 100:.2f}%)")

## 6. Top-10 결과 출력

확률이 높은 순서대로 상위 10개 클래스를 함께 출력합니다.

In [ ]:
# 확률 기준 상위 10개 클래스를 가져옵니다.
top_k = 10
top_probabilities, top_class_ids = torch.topk(probabilities, k=top_k)

print("Top-10 예측 결과")
print("-" * 72)
print(f"{'rank':>4} | {'class_id':>8} | {'probability':>11} | label")
print("-" * 72)

for rank, (class_id, probability) in enumerate(zip(top_class_ids.tolist(), top_probabilities.tolist()), start=1):
    label = model.config.id2label[class_id]
    print(f"{rank:>4} | {class_id:>8} | {probability:>10.4f} | {label}")

## 7. Kaggle Input에 있는 내 이미지로 추론하기

이미 학습된 `microsoft/resnet-18` 모델에 Kaggle Input의 이미지를 넣어 결과를 확인합니다. 이 과정은 추가 학습이 아니라 추론입니다.

In [ ]:
from PIL import Image

# Kaggle Input에 있는 이미지 경로를 넣습니다.
# 예: /kaggle/input/datasets/kosukmin/data-bus/KakaoTalk_20260423_111222625.jpg
image_path = "/kaggle/input/datasets/kosukmin/data-bus/KakaoTalk_20260423_111222625.jpg"

# 이미지를 RGB로 변환해야 ResNet 입력 채널과 맞습니다.
image = Image.open(image_path).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title("Kaggle Input Image")
plt.show()

# 전처리 후 모델과 같은 장치로 이동합니다.
inputs = processor(images=image, return_tensors="pt")
inputs = {key: value.to(device) for key, value in inputs.items()}

# 이미 학습된 모델로 추론합니다.
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# outputs.logits shape: [1, 1000]
# 이미지 1장에 대한 확률만 보기 위해 [0]으로 배치 차원을 제거합니다.
probabilities = torch.softmax(outputs.logits, dim=-1)[0]

# 가장 높은 확률의 클래스입니다.
predicted_class_id = int(torch.argmax(probabilities).item())
predicted_label = model.config.id2label[predicted_class_id]
predicted_probability = float(probabilities[predicted_class_id].item())

print("가장 높은 확률의 예측 결과")
print(f"class id   : {predicted_class_id}")
print(f"label      : {predicted_label}")
print(f"probability: {predicted_probability:.4f} ({predicted_probability * 100:.2f}%)")

# ImageNet은 클래스가 1000개라 top-10을 그대로 볼 수 있습니다.
top_k = 10
top_probabilities, top_class_ids = torch.topk(probabilities, k=top_k)

print("\nTop-10 예측 결과")
print("-" * 72)
print(f"{'rank':>4} | {'class_id':>8} | {'probability':>11} | label")
print("-" * 72)

for rank, (class_id, probability) in enumerate(zip(top_class_ids.tolist(), top_probabilities.tolist()), start=1):
    label = model.config.id2label[class_id]
    print(f"{rank:>4} | {class_id:>8} | {probability:>10.4f} | {label}")

## 전체 흐름을 함수로 재사용하기

아래 함수는 PIL 이미지 하나를 받아 가장 높은 확률의 클래스와 top-k 결과를 반환합니다.

In [ ]:
def classify_image(image, model, processor, device, top_k=10):
    """PIL 이미지를 입력받아 ResNet 이미지 분류 결과를 반환합니다."""
    image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits[0], dim=-1)
    top_probabilities, top_class_ids = torch.topk(probabilities, k=top_k)

    results = []
    for class_id, probability in zip(top_class_ids.tolist(), top_probabilities.tolist()):
        results.append({
            "class_id": class_id,
            "label": model.config.id2label[class_id],
            "probability": probability,
        })

    return results[0], results


best_result, top_10_results = classify_image(image, model, processor, device, top_k=10)

print("Best result:")
print(best_result)

print("\nTop-10 results:")
for rank, result in enumerate(top_10_results, start=1):
    print(f"{rank:>2}. {result['label']} - {result['probability']:.4f}")